# AeroPulse Propagation — 07: Model Evaluation & Promotion

Rolling-origin folds, geographic-block holdout, per-station and per-regime
metrics, extreme-event recall, bias, and a promotion decision that requires
evidence for every gate.

## Objective

Improvement plan sections 23-28, 30 and 40.

A single chronological split and an aggregate MAE are enough to build a
model and nowhere near enough to ship one. This notebook asks the questions
that decide whether a horizon is safe to serve:

- does the skill **hold across time**, or was one lucky test window carrying it? (rolling-origin, section 23)
- does it hold in a **region the model has never seen**? (geographic blocks, section 24)
- which **stations** does it fail at? (section 25)
- does it work during **episodes**, or only on clean air? (sections 26-27)
- does it **underpredict** the events an alerting product exists to catch? (section 28)
- do all of the above clear the **promotion gates**? (section 40)

Every gate needs evidence. A gate with no measurement fails — an unmeasured
property is not a satisfied one.

In [1]:
# ============================================================================
# SETUP — everything reusable lives in propagation_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import propagation_toolkit as pt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

cfg = pt.load_config()
HORIZONS = cfg.horizons
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast
horizons: [1, 3, 6, 12, 24, 48] | schema 2.0.0
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
import joblib

hybrid_stage = cfg.stage_dir("hybrid")
hybrid_report = json.loads((hybrid_stage / "hybrid_report.json").read_text())
BEST_PARAMS = hybrid_report["best_params"]
WINNER = hybrid_report["winner_family"]
BEST_BASELINE = {int(k): v for k, v in hybrid_report["anchor_baselines"].items()}
horizon_metrics = pd.read_csv(hybrid_stage / "horizon_metrics.csv")
models = joblib.load(hybrid_stage / "residual_models.joblib")

uncertainty_stage = cfg.stage_dir("uncertainty")
quantiles = joblib.load(uncertainty_stage / "quantile_bundles.joblib")
coverage_table = pd.read_csv(uncertainty_stage / "test_coverage.csv")

EVAL_CAP = cfg.max_train_rows or 350_000
print("family", WINNER, "| params", BEST_PARAMS)

df = pd.read_parquet(cfg.feature_dir / "propagation_features.parquet")
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
print("rows", f"{len(df):,}")

family catboost | params {'subsample': 0.8, 'reg_lambda': 5, 'reg_alpha': 0.1, 'num_leaves': 31, 'n_estimators': 300, 'min_child_samples': 150, 'max_depth': 12, 'learning_rate': 0.05, 'colsample_bytree': 0.6}


rows 1,705,252


In [3]:
def subsample(frame, cap):
    if cap is None or len(frame) <= cap:
        return frame
    return frame.iloc[(np.arange(cap) * (len(frame) / cap)).astype(int)]


_PREPARED = {}


def prepare(h):
    """Cached: every section below needs the same baseline-anchored frame."""
    if h in _PREPARED:
        return _PREPARED[h]
    target_col = f"target_pm25_t_plus_{h}h"
    work = df.dropna(subset=[target_col, "pm25"]).copy()
    baselines = pt.baseline_forecasts(work, h)
    for name in pt.BASELINE_NAMES:
        work[f"bl_{name}"] = baselines[name]
    work["baseline_forecast"] = work[f"bl_{BEST_BASELINE[h]}"].fillna(work["pm25"])
    _PREPARED[h] = (work, target_col)
    return work, target_col

## 1. Rolling-origin validation (section 23)

Expanding training window, four sequential validation blocks, each purged by
`h` hours. One good test window proves nothing; four consecutive ones is
evidence.

In [4]:
fold_rows = []
for h in HORIZONS:
    work, target_col = prepare(h)
    folds = pt.rolling_origin_folds(work, h, n_folds=cfg.rolling_folds)
    for train_idx, valid_idx, meta in folds:
        train = subsample(work.loc[train_idx], EVAL_CAP)
        valid = subsample(work.loc[valid_idx], 120_000)
        features = pt.select_features(train, horizon=h)
        model = pt.ResidualModel.fit(train, features, h, kind=WINNER, params=BEST_PARAMS)
        pred = model.predict(valid)
        y = valid[target_col].to_numpy(dtype=float)
        metrics = pt.regression_metrics(y, pred, valid["baseline_forecast"].to_numpy(dtype=float))
        fold_rows.append({
            "horizon_h": h, "fold": meta["fold"],
            "valid_start": meta["valid_start"][:10], "valid_end": meta["valid_end"][:10],
            "n_train": len(train), "n_valid": metrics["n"],
            "mae": round(metrics["mae"], 3),
            "baseline_mae": round(metrics["mae_reference"], 3),
            "skill": round(metrics["skill"], 4),
            "skill_vs_persistence": round(
                pt.skill(metrics["mae"], float(np.abs(valid["pm25"].to_numpy(dtype=float) - y).mean())), 4),
        })
    done = [r for r in fold_rows if r["horizon_h"] == h]
    print(f"h={h:>2}  folds {len(done)}  skill "
          + " ".join(f"{r['skill']:+.3f}" for r in done))

folds_table = pd.DataFrame(fold_rows)

h= 1  folds 4  skill +0.166 +0.147 +0.083 +0.035


h= 3  folds 4  skill +0.123 +0.141 +0.096 +0.063


h= 6  folds 4  skill +0.153 +0.158 +0.107 +0.073


h=12  folds 4  skill +0.133 +0.137 +0.108 +0.089


h=24  folds 4  skill +0.051 +0.087 +0.096 +0.082


h=48  folds 4  skill +0.087 +0.114 +0.083 +0.079


In [5]:
fold_summary = (folds_table.groupby("horizon_h")
    .agg(folds=("skill", "size"),
         mean_skill=("skill", "mean"),
         median_skill=("skill", "median"),
         std_skill=("skill", "std"),
         worst_skill=("skill", "min"),
         folds_positive=("skill", lambda s: int((s > 0).sum())))
    .round(4).reset_index())
print(fold_summary.to_string(index=False))

 horizon_h  folds  mean_skill  median_skill  std_skill  worst_skill  folds_positive
         1      4      0.1078        0.1151     0.0604       0.0346               4
         3      4      0.1056        0.1092     0.0339       0.0631               4
         6      4      0.1226        0.1299     0.0400       0.0731               4
        12      4      0.1168        0.1207     0.0226       0.0888               4
        24      4      0.0792        0.0849     0.0197       0.0509               4
        48      4      0.0907        0.0849     0.0162       0.0785               4


## 2. Geographic-block holdout (section 24)

Stations clustered by coordinates; train on four blocks, test on the fifth.
A random station split leaks the local pollution climate — the held-out
station usually has a neighbour a few km away sitting in the training set.
This asks the question the product actually cares about: can AeroPulse expand
to a region it has never seen?

In [6]:
blocks = pt.geographic_blocks(df, n_blocks=cfg.geographic_blocks, random_state=cfg.random_state)
print(blocks.groupby("geo_block").agg(
    stations=("location_id", "size"), lat=("block_lat", "first"), lon=("block_lon", "first")
).round(2).to_string())

block_of = dict(zip(blocks["location_id"], blocks["geo_block"]))
# Every horizon, not a sample of them. The promotion gate fails on missing
# evidence by design, so measuring only two horizons would withhold the other
# four for a reason that has nothing to do with their quality.
GEO_HORIZONS = list(HORIZONS)

geo_rows = []
for h in GEO_HORIZONS:
    work, target_col = prepare(h)
    work["geo_block"] = work["location_id"].map(block_of)
    for block in sorted(blocks["geo_block"].unique()):
        train = subsample(work[work["geo_block"] != block], EVAL_CAP)
        held = subsample(work[work["geo_block"] == block], 120_000)
        if len(held) < 2000:
            continue
        features = pt.select_features(train, horizon=h)
        model = pt.ResidualModel.fit(train, features, h, kind=WINNER, params=BEST_PARAMS)
        pred = model.predict(held)
        y = held[target_col].to_numpy(dtype=float)
        metrics = pt.regression_metrics(y, pred, held["baseline_forecast"].to_numpy(dtype=float))
        geo_rows.append({
            "horizon_h": h, "held_block": int(block),
            "stations": int(held["location_id"].nunique()),
            "n": metrics["n"],
            "mae": round(metrics["mae"], 3),
            "baseline_mae": round(metrics["mae_reference"], 3),
            "skill": round(metrics["skill"], 4),
        })
        print(f"  h={h:>2} block {block}  {held['location_id'].nunique():>3} stations  "
              f"MAE {metrics['mae']:.2f}  skill {metrics['skill']:+.4f}")

geo_table = pd.DataFrame(geo_rows)
print()
print(geo_table.to_string(index=False))

           stations        lat        lon
geo_block                                
0                27  19.010000  79.169998
1                28  27.860001  77.500000
2                31  23.610001  86.830002
3                42  18.920000  73.660004
4                21  24.639999  74.230003


  h= 1 block 0   27 stations  MAE 4.08  skill +0.0636


  h= 1 block 1   28 stations  MAE 7.62  skill +0.1159


  h= 1 block 2   31 stations  MAE 5.55  skill +0.0916


  h= 1 block 3   42 stations  MAE 4.49  skill +0.0696


...

  h=48 block 0   27 stations  MAE 9.37  skill +0.1381


  h=48 block 1   28 stations  MAE 17.57  skill +0.1320


  h=48 block 2   31 stations  MAE 14.14  skill +0.1269


  h=48 block 3   42 stations  MAE 10.55  skill +0.1235


  h=48 block 4   21 stations  MAE 12.52  skill +0.1386

 horizon_h  held_block  stations      n    mae  baseline_mae  skill
         1           0        27 120000  4.075         4.352 0.0636
         1           1        28 120000  7.622         8.621 0.1159
         1           2        31 120000  5.551         6.110 0.0916
         1           3        42 120000  4.488         4.824 0.0696
         1           4        21 120000  5.018         5.833 0.1397
         3           0        27 120000  6.765         7.493 0.0971
         3           1        28 120000 13.110        14.787 0.1134
         3           2        31 120000  9.750        11.045 0.1173
         3           3        42 120000  7.410         8.260 0.1030
         3           4        21 120000  9.163        10.432 0.1216
         6           0        27 120000  7.760         8.642 0.1021
         6           1        28 120000 15.093        17.601 0.1425
         6           2        31 120000 11.505        13.143

## 3. Per-station metrics (section 25)

In [7]:
station_frames = []
for h in HORIZONS:
    work, target_col = prepare(h)
    _, _, test = pt.chrono_split(work, h, cfg.train_fraction, cfg.validation_fraction)
    test = test.copy()
    test["prediction"] = models[h].predict(test)
    station_frames.append(
        pt.station_metrics(test, target_col, "prediction", "baseline_forecast", h)
    )
stations = pd.concat(station_frames, ignore_index=True)

regression_rows = []
for h in HORIZONS:
    block = stations[stations["horizon_h"] == h]
    worse = block[block["skill"] <= 0]
    regression_rows.append({
        "horizon_h": h,
        "stations": len(block),
        "stations_worse_than_baseline": len(worse),
        "regression_pct": round(len(worse) / len(block) * 100, 1),
        "median_station_skill": round(float(block["skill"].median()), 4),
        "p10_station_skill": round(float(block["skill"].quantile(0.10)), 4),
        "worst_station_skill": round(float(block["skill"].min()), 4),
    })
station_summary = pd.DataFrame(regression_rows)
print(station_summary.to_string(index=False))

print("\nworst 8 station/horizon pairs:")
print(stations.nsmallest(8, "skill")[
    ["location_id", "horizon_h", "n", "mae", "mae_reference", "skill", "bias"]
].to_string(index=False))

 horizon_h  stations  stations_worse_than_baseline  regression_pct  median_station_skill  p10_station_skill  worst_station_skill
         1       149                            30            20.1                0.0401            -0.0481              -1.1491
         3       149                            17            11.4                0.0571            -0.0020              -0.4906
         6       149                            19            12.8                0.0638            -0.0657              -0.9158
        12       149                            26            17.4                0.0734            -0.0533              -1.1108
        24       149                            33            22.1                0.0807            -0.1442              -1.1055
        48       149                            42            28.2                0.0838            -0.2472              -1.8488

worst 8 station/horizon pairs:
 location_id  horizon_h    n       mae  mae_reference     skill  

## 4. Per-regime metrics (section 26)

In [8]:
regime_frames = []
for h in HORIZONS:
    work, target_col = prepare(h)
    _, _, test = pt.chrono_split(work, h, cfg.train_fraction, cfg.validation_fraction)
    test = test.copy()
    test["prediction"] = models[h].predict(test)
    regime_frames.append(
        pt.regime_metrics(test, target_col, "prediction", "baseline_forecast", h)
    )
regimes = pd.concat(regime_frames, ignore_index=True)

concentration = regimes[regimes["group_kind"] == "concentration"]
print("by concentration band:")
print(concentration.pivot_table(index="group", columns="horizon_h", values="skill").round(3).to_string())
print("\nMAE by concentration band:")
print(concentration.pivot_table(index="group", columns="horizon_h", values="mae").round(1).to_string())
print("\nby dynamic regime (skill):")
print(regimes[regimes["group_kind"] == "dynamic"]
      .pivot_table(index="group", columns="horizon_h", values="skill").round(3).to_string())

by concentration band:
horizon_h     1      3      6      12     24     48
group                                              
120-250    0.127  0.265  0.358  0.592  0.441  0.493
30-60      0.042  0.043  0.053  0.080  0.072  0.076
60-120     0.072  0.113  0.180  0.290  0.246  0.315
<30        0.012  0.051  0.037  0.021  0.008 -0.024
>250       0.190  0.310  0.470  0.731  0.629  0.740

MAE by concentration band:
horizon_h     1      3      6     12     24    48
group                                            
120-250     60.9   53.4   46.3  29.1   45.7  42.1
30-60        5.7    8.7    9.4  10.1   10.5  11.5
60-120      19.2   23.4   22.8  19.8   25.0  24.3
<30          3.0    4.9    5.7   6.3    6.4   7.4
>250       162.3  143.3  117.0  61.4  106.5  79.3

by dynamic regime (skill):
horizon_h               1      3      6      12     24     48
group                                                        
falling              0.030  0.056  0.057  0.055  0.062  0.057
fire_influenced      

## 5. Extreme-event recall and bias (sections 27-28)

In [9]:
event_rows, bias_rows = [], []
for h in HORIZONS:
    work, target_col = prepare(h)
    _, _, test = pt.chrono_split(work, h, cfg.train_fraction, cfg.validation_fraction)
    pred = models[h].predict(test)
    y = test[target_col].to_numpy(dtype=float)
    baseline = test["baseline_forecast"].to_numpy(dtype=float)

    for threshold in (100.0, pt.EVENT_THRESHOLD_UGM3):
        model_event = pt.event_metrics(y, pred, threshold)
        baseline_event = pt.event_metrics(y, baseline, threshold)
        event_rows.append({
            "horizon_h": h, "threshold": threshold,
            "prevalence": round(model_event["prevalence"], 5),
            "model_precision": round(model_event["precision"], 3),
            "model_recall": round(model_event["recall"], 3),
            "model_f1": round(model_event["f1"], 3),
            "model_far": round(model_event["false_alarm_rate"], 5),
            "baseline_recall": round(baseline_event["recall"], 3),
            "baseline_precision": round(baseline_event["precision"], 3),
        })
    bias = pt.bias_metrics(y, pred)
    bias["horizon_h"] = h
    bias_rows.append(bias)

events = pd.DataFrame(event_rows)
bias_table = pd.DataFrame(bias_rows)[
    ["horizon_h", "mean_error", "median_error", "underprediction_pct",
     "high_pm25_n", "high_pm25_mean_error", "high_pm25_underprediction_pct"]
].round(3)
print("exceedance detection:")
print(events.to_string(index=False))
print("\nbias:")
print(bias_table.to_string(index=False))

exceedance detection:
 horizon_h  threshold  prevalence  model_precision  model_recall  model_f1  model_far  baseline_recall  baseline_precision
         1      100.0     0.00538            0.638         0.540     0.585    0.00165            0.578               0.587
         1      150.0     0.00211            0.589         0.434     0.499    0.00064            0.518               0.535
         3      100.0     0.00537            0.527         0.190     0.279    0.00092            0.239               0.344
         3      150.0     0.00212            0.425         0.094     0.154    0.00027            0.154               0.252
         6      100.0     0.00525            0.469         0.103     0.169    0.00062            0.192               0.270
         6      150.0     0.00208            0.182         0.020     0.036    0.00019            0.121               0.175
        12      100.0     0.00532            0.379         0.044     0.079    0.00039            0.146               

### Alerting on P90 instead of the point forecast

A point forecast trained on squared error predicts the conditional *mean*, so
on the rare rows where PM2.5 is about to spike it will sit far below the
actual value almost every time. Judging "will it exceed 150" by that number
measures the wrong thing.

The P90 bound from notebook 06 is the right instrument for an exceedance
question: it asks whether a high outcome is *plausible*, not whether it is
the expected one. Same threshold, same rows, different statistic.

In [10]:
p90_rows = []
for h in HORIZONS:
    work, target_col = prepare(h)
    _, _, test = pt.chrono_split(work, h, cfg.train_fraction, cfg.validation_fraction)
    bundle = quantiles.get(h)
    if bundle is None:
        continue
    q = bundle.predict(test)
    y = test[target_col].to_numpy(dtype=float)
    point = models[h].predict(test)
    for threshold in (100.0, pt.EVENT_THRESHOLD_UGM3):
        point_event = pt.event_metrics(y, point, threshold)
        p90_event = pt.event_metrics(y, q.iloc[:, -1].to_numpy(), threshold)
        p90_rows.append({
            "horizon_h": h, "threshold": threshold,
            "point_recall": round(point_event["recall"], 3),
            "p90_recall": round(p90_event["recall"], 3),
            "point_precision": round(point_event["precision"], 3),
            "p90_precision": round(p90_event["precision"], 3),
            "point_far": round(point_event["false_alarm_rate"], 5),
            "p90_far": round(p90_event["false_alarm_rate"], 5),
            "p90_f1": round(p90_event["f1"], 3),
        })
p90_events = pd.DataFrame(p90_rows)
print(p90_events.to_string(index=False))

 horizon_h  threshold  point_recall  p90_recall  point_precision  p90_precision  point_far  p90_far  p90_f1
         1      100.0         0.540       0.727            0.638          0.433    0.00165  0.00515   0.543
         1      150.0         0.434       0.645            0.589          0.400    0.00064  0.00204   0.494
         3      100.0         0.190       0.462            0.527          0.264    0.00092  0.00695   0.336
         3      150.0         0.094       0.307            0.425          0.242    0.00027  0.00204   0.270
         6      100.0         0.103       0.384            0.469          0.239    0.00062  0.00644   0.295
         6      150.0         0.020       0.209            0.182          0.185    0.00019  0.00191   0.196
        12      100.0         0.044       0.290            0.379          0.222    0.00039  0.00542   0.251
        12      150.0         0.006       0.118            0.115          0.143    0.00010  0.00149   0.129
        24      100.0       

## 6. Latency benchmark (section 48)

In [11]:
latency_rows = []
for h in HORIZONS:
    work, _ = prepare(h)
    _, _, test = pt.chrono_split(work, h, cfg.train_fraction, cfg.validation_fraction)
    single = test.head(1)
    batch = test.head(1000)
    models[h].predict(single)   # warm

    t0 = time.perf_counter()
    for _ in range(50):
        models[h].predict(single)
    single_ms = (time.perf_counter() - t0) / 50 * 1000

    t0 = time.perf_counter()
    models[h].predict(batch)
    batch_ms = (time.perf_counter() - t0) / len(batch) * 1000
    latency_rows.append({
        "horizon_h": h,
        "single_row_ms": round(single_ms, 3),
        "batched_ms_per_row": round(batch_ms, 5),
        "within_200ms_sla": bool(single_ms < 200),
    })
latency = pd.DataFrame(latency_rows)
print(latency.to_string(index=False))

 horizon_h  single_row_ms  batched_ms_per_row  within_200ms_sla
         1          8.630             0.00956              True
         3          7.500             0.00948              True
         6          7.346             0.00995              True
        12          7.604             0.00980              True
        24          7.565             0.01613              True
        48          7.410             0.00935              True


## 7. Promotion decision (section 40)

Nine gates, each requiring measured evidence. `evaluate_promotion` fails any
gate whose evidence is missing.

In [12]:
decisions, rows = {}, []
for h in HORIZONS:
    metrics_row = horizon_metrics[horizon_metrics["horizon_h"] == h].iloc[0]
    fold_row = fold_summary[fold_summary["horizon_h"] == h].iloc[0]
    station_row = station_summary[station_summary["horizon_h"] == h].iloc[0]
    coverage_row = coverage_table[coverage_table["horizon_h"] == h].iloc[0]
    event_row = events[(events["horizon_h"] == h) & (events["threshold"] == pt.EVENT_THRESHOLD_UGM3)].iloc[0]
    bias_row = bias_table[bias_table["horizon_h"] == h].iloc[0]
    geo_block = geo_table[geo_table["horizon_h"] == h]
    latency_row = latency[latency["horizon_h"] == h].iloc[0]

    evidence = {
        "test_skill": float(metrics_row["test_skill_vs_persistence"]),
        "skill_vs_best_baseline": float(metrics_row["test_skill"]),
        "folds_positive": int(fold_row["folds_positive"]),
        "station_regression_pct": float(station_row["regression_pct"]),
        "geo_block_min_skill": float(geo_block["skill"].min()) if len(geo_block) else None,
        "p80_coverage": float(coverage_row["test_coverage"]),
        "event_recall": float(event_row["model_recall"]) if np.isfinite(event_row["model_recall"]) else 0.0,
        "high_pm25_underprediction_pct": float(bias_row["high_pm25_underprediction_pct"]),
        "latency_ms": float(latency_row["single_row_ms"]),
    }
    decision = pt.evaluate_promotion(evidence)
    decision["evidence"] = evidence
    decisions[h] = decision
    rows.append({
        "horizon_h": h,
        "promote": decision["promote"],
        "gates_passed": sum(c["passed"] for c in decision["checks"]),
        "gates_total": len(decision["checks"]),
        "failed": ", ".join(decision["failed_gates"]) or "-",
    })

promotion = pd.DataFrame(rows)
print(promotion.to_string(index=False))

 horizon_h  promote  gates_passed  gates_total                                         failed
         1    False             7            9          beats_persistence, station_regression
         3    False             7            9                     event_recall, extreme_bias
         6    False             7            9                     event_recall, extreme_bias
        12    False             7            9                     event_recall, extreme_bias
        24    False             6            9 station_regression, event_recall, extreme_bias
        48    False             6            9 station_regression, event_recall, extreme_bias


In [13]:
for h in HORIZONS:
    decision = decisions[h]
    print(f"\nh={h}h  ->  {'PROMOTE' if decision['promote'] else 'WITHHOLD'}")
    for check in decision["checks"]:
        mark = "pass" if check["passed"] else "FAIL"
        value = check["value"]
        shown = f"{value:.4f}" if isinstance(value, float) else str(value)
        print(f"   [{mark}] {check['gate']:<28} {shown:>10}   ({check['requirement']})")


h=1h  ->  WITHHOLD
   [FAIL] beats_persistence                0.0365   (test skill vs persistence > 0.05)
   [pass] beats_best_baseline              0.0365   (skill vs strongest baseline > 0.0)
   [pass] rolling_folds_positive           4.0000   (>= 3 rolling-origin folds with positive skill)
   [FAIL] station_regression              20.1000   (<= 20.0% of stations worse than persistence)
   [pass] geographic_generalization        0.0636   (positive skill on every held-out geographic block)
   [pass] uncertainty_coverage             0.7730   (calibrated P10-P90 coverage >= 0.7)
   [pass] event_recall                     0.4340   (exceedance recall >= 0.2)
   [pass] extreme_bias                    84.1090   (<= 85.0% underprediction on high-PM2.5 rows)
   [pass] latency                          8.6300   (<= 200.0 ms per forecast)

h=3h  ->  WITHHOLD
   [pass] beats_persistence                0.1292   (test skill vs persistence > 0.05)
   [pass] beats_best_baseline              0.0662  

## 8. Save

In [14]:
stage = cfg.stage_dir("evaluation")
folds_table.to_csv(stage / "rolling_origin_folds.csv", index=False)
fold_summary.to_csv(stage / "rolling_origin_summary.csv", index=False)
geo_table.to_csv(stage / "geographic_blocks.csv", index=False)
blocks.to_csv(stage / "station_blocks.csv", index=False)
stations.to_csv(stage / "station_metrics.csv", index=False)
station_summary.to_csv(stage / "station_summary.csv", index=False)
regimes.to_csv(stage / "regime_metrics.csv", index=False)
events.to_csv(stage / "event_metrics.csv", index=False)
p90_events.to_csv(stage / "event_metrics_p90.csv", index=False)
bias_table.to_csv(stage / "bias_metrics.csv", index=False)
latency.to_csv(stage / "latency.csv", index=False)

certification = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "git_commit": pt.git_commit(),
    "model_family": WINNER,
    "params": BEST_PARAMS,
    "gates": pt.PROMOTION_GATES,
    "decisions": {str(h): decisions[h] for h in HORIZONS},
    "promoted": [h for h in HORIZONS if decisions[h]["promote"]],
    "withheld": [h for h in HORIZONS if not decisions[h]["promote"]],
    "rolling_origin": fold_summary.to_dict(orient="records"),
    "geographic": geo_table.to_dict(orient="records"),
    "stations": station_summary.to_dict(orient="records"),
    "events": events.to_dict(orient="records"),
    "bias": bias_table.to_dict(orient="records"),
    "latency": latency.to_dict(orient="records"),
    "p90_events": p90_events.to_dict(orient="records"),
}
(stage / "certification_report.json").write_text(json.dumps(certification, indent=2, default=str))

print("07 COMPLETE")
print("  promoted:", certification["promoted"])
print("  withheld:", certification["withheld"])
print("Next: 08_propagation_model_packaging.ipynb")

07 COMPLETE
  promoted: []
  withheld: [1, 3, 6, 12, 24, 48]
Next: 08_propagation_model_packaging.ipynb


## Findings

**Verdict: all six horizons withheld.** Not because the model is bad at what
it was trained to do — it beats its baseline everywhere and generalises to
unseen regions — but because it fails the two gates that matter for an air
quality product. The detail is worth reading before anyone reaches for the
threshold knobs.

### What passes, convincingly

**Rolling-origin: 4/4 folds positive at every horizon.** Mean skill vs the
anchor baseline runs 0.079 (24 h) to 0.123 (6 h), with fold-to-fold standard
deviation of 0.016-0.060. The single test window in notebook 05 was not a
lucky draw.

One caveat visible in the folds: skill **decays monotonically across time**
at the short horizons — h=1 goes +0.166, +0.147, +0.083, +0.035 across the
four folds. Whatever the model learned early is worth less by the last fold.
That is a retraining-cadence signal, and it argues for the weekly policy in
the deployment manifest rather than against it.

**Geographic generalisation: passes on every block, every horizon.** Five
k-means blocks over station coordinates, train on four and test on the fifth:

| horizon | worst block skill | best block skill |
|---|---|---|
| 1 h | +0.064 | +0.140 |
| 6 h | +0.102 | +0.147 |
| 24 h | +0.096 | +0.113 |
| 48 h | +0.124 | +0.139 |

No block is negative. Held-out MAE varies a lot by region (7.8 to 15.1 at
6 h) but the *skill* is stable, meaning the model is not memorising station
identity — it transfers to regions it has never seen. This is the strongest
result in the notebook and it is what makes geographic expansion credible.

(First run measured only h=6 and h=24, so four horizons failed this gate for
lack of evidence rather than lack of skill. Re-run across all horizons; the
"missing evidence fails" rule is right, but it has to be pointed at a
measurement that was actually attempted.)

**Latency: 7.3-8.6 ms per single-row forecast**, against a 200 ms SLA.
Batched, it is ~0.01 ms/row. Not a constraint.

### What fails, and why it is real

**Exceedance recall collapses with horizon.** At the 150 µg/m³ threshold:

| horizon | model recall | model precision | baseline recall |
|---|---|---|---|
| 1 h | 0.434 | 0.589 | 0.518 |
| 3 h | 0.094 | 0.425 | 0.154 |
| 6 h | 0.020 | 0.182 | 0.121 |
| 12 h | 0.006 | 0.115 | 0.045 |
| 48 h | 0.000 | — | 0.070 |

Read the last column. **Beyond 1 h the model catches fewer exceedances than
the baseline it beats on MAE.** At 6 h the baseline finds 12.1% of them and
the model finds 2.0%. At 48 h the model finds none at all.

**Bias explains it.** Mean error on rows where PM2.5 ≥ 120: **-79.8 at 1 h,
-162.1 at 6 h, -185.9 at 48 h µg/m³.** Underprediction rate on those rows is
97-100% beyond 1 h. The model is not slightly conservative during episodes;
it barely participates in them.

This is squared-error regression behaving exactly as specified. Exceedances
are 0.2% of rows; predicting the conditional mean means predicting "not an
episode" and absorbing a small MAE penalty on a rare event. Aggregate MAE
improves. The product gets worse.

Plan section 28 anticipated precisely this — "if the model systematically
underpredicts extreme events, it should not be promoted even if aggregate
MAE looks good" — and the gate did its job.

**Note on the `extreme_bias` gate itself:** at ≤85% underprediction it is
close to unpassable for *any* conditional-mean forecaster on a heavy right
tail, and only h=1 (84.1%) squeaks through. The threshold is arguably badly
specified. But the -162 µg/m³ mean error is not a threshold artifact, so
loosening the gate would hide a real defect rather than correct a
measurement error. Leave it, and fix the model.

### The fix, measured rather than proposed

Exceedance is a question about whether a high outcome is *plausible*, not
whether it is *expected*. Asking the P90 bound instead of the point forecast:

| horizon | threshold | point recall | **P90 recall** | point FAR | P90 FAR |
|---|---|---|---|---|---|
| 1 h | 100 | 0.540 | **0.727** | 0.17% | 0.52% |
| 3 h | 150 | 0.094 | **0.307** | 0.03% | 0.20% |
| 6 h | 150 | 0.020 | **0.209** | 0.02% | 0.19% |
| 12 h | 150 | 0.006 | **0.118** | 0.01% | 0.15% |
| 24 h | 150 | 0.010 | **0.152** | 0.01% | 0.14% |

Recall improves by **2x to 20x**, and the false-alarm rate stays under 0.7%
in every case. On this evidence P90 alerting clears the 0.20 recall gate at
1 h, 3 h and 6 h — the horizons an alerting product actually needs.

The quantile models are already trained, already calibrated (notebook 06),
and already inside the registered bundle. This costs nothing to adopt.

### Secondary findings

**Skill is highest exactly where the baseline is worst.** By concentration
band at 12 h: +0.021 below 30 µg/m³, +0.290 in 60-120, +0.592 in 120-250,
+0.731 above 250. The model *is* adding value during episodes in MAE terms —
it is closer than the baseline while still being far too low in absolute
terms. Both statements are true and neither cancels the other.

**Transport-dominated rows are where it adds nothing:** skill +0.004 at 24 h
and **-0.024 at 48 h**, the only negative cell in the table. Rising and
stagnant regimes are where it earns its keep (+0.26 and +0.29 at 12 h). Given
notebook 05's ablation put the transport block at 0.24% of MAE, the picture
is consistent: this pipeline models local dynamics well and genuine
long-range transport poorly.

**Station-level regression grows with horizon:** 20.1% of stations are worse
than the baseline at 1 h, 11.4% at 3 h, rising to 28.2% at 48 h. The worst
offenders repeat across horizons — stations 358467, 3409395 and 270729
appear in seven of the eight worst pairs, with skill down to -1.85. Three
stations, not a diffuse problem: worth inspecting their raw series before
concluding anything about the model.